# ARC-v0.5 — Sealed FEVER DEV Confirmation

## Confirmatory status
ARC-v0.1–v0.4 used FEVER FIT for discovery and mechanism development.
ARC-v0.5 uses the untouched **FEVER DEV split** only.

Before any DEV retrieval, the notebook writes a sealed protocol manifest to Drive.
No DEV-based hyperparameter selection is allowed and TEST remains untouched.

## Preregistered primary hypotheses

Starting from the same original query \(q_0\):

### H1 Query-state amplification
\[
E[\mathrm{slope}_t(1-\cos(q_t^{PQ32},q_t^{SQ8}))]>0.
\]

### H2 Additional candidate-set amplification
\[
\widetilde D_C(t)=D_C(t)-D_C(0)
\]
and
\[
E[\mathrm{slope}_t\widetilde D_C(t)]>0.
\]

### H3 Utility-gap amplification
\[
E[\mathrm{slope}_t |nDCG@10_{SQ8}(t)-nDCG@10_{PQ32}(t)|]>0.
\]

### H4 Feedback-source intervention
At the final round:
\[
E[nDCG@10_B-nDCG@10_A]>0,
\]
where
\[
A=PQ32\ search\rightarrow PQ32\ feedback,
\]
\[
B=PQ32\ search\rightarrow SQ8\ feedback.
\]

## Frozen design
- all 6,666 FEVER DEV queries
- BAAI/bge-small-en-v1.5
- 5,416,568-document corpus
- IVF-PQ32 vs IVF-SQ8
- nlist=4096, nprobe=64
- Top-100 retrieval
- 4 feedback rounds
- exactly the feedback configurations frozen in ARC-v0.3
- query is the primary independent unit
- query-level bootstrap CI
- one-sided paired sign-flip randomization
- Holm correction jointly across H1–H4
- full confirmation passes only if all four endpoints pass


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, gc, sys, subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260816
DIM = 384
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4
NLIST = 4096
NPROBE = 64

BOOTSTRAP_SAMPLES = 20_000
RANDOMIZATION_SAMPLES = 20_000
RESAMPLE_CHUNK = 250

ROOT = Path("/content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1")
CORPUS_MEMMAP = ROOT / "stage1/corpus_embeddings.float16.memmap"
QUERY_EMB = ROOT / "stage1/query_embeddings_v2.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
DEV_QRELS = ROOT / "stage2/dev_qrels_rows.csv"

ARC_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
CACHE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache")

CONFIRM_ROOT = ARC_ROOT / "sealed-fever-dev-confirmation-v05"
CONFIRM_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = CONFIRM_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

print("Output:", OUT)


Output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214


In [3]:
def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

run([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu==1.12.0", "psutil", "pyarrow"
])

import faiss, psutil

PROCESS = psutil.Process(os.getpid())

def ram_status(label=""):
    vm = psutil.virtual_memory()
    rss = PROCESS.memory_info().rss / 1024**3
    print(
        f"[RAM] {label:<28} "
        f"RSS={rss:6.2f} GB | "
        f"available={vm.available/1024**3:6.2f} GB | "
        f"used={vm.percent:5.1f}%"
    )

print("Faiss:", faiss.__version__)
ram_status("startup")


$ /usr/bin/python3 -m pip install -q faiss-cpu==1.12.0 psutil pyarrow
Faiss: 1.12.0
[RAM] startup                      RSS=  0.17 GB | available= 11.39 GB | used= 10.1%


## 1. Recover frozen choices from completed FIT lineage


In [4]:
V03_RUNS = sorted(
    [p for p in ARC_ROOT.glob("error-amplification-v03-*")
     if (p/"report.json").is_file()],
    key=lambda p:p.stat().st_mtime,
)
V04_RUNS = sorted(
    [p for p in ARC_ROOT.glob("statistical-mechanism-audit-v04-*")
     if (p/"report.json").is_file()],
    key=lambda p:p.stat().st_mtime,
)

if not V03_RUNS:
    raise FileNotFoundError("Completed ARC-v0.3 report not found.")
if not V04_RUNS:
    raise FileNotFoundError("Completed ARC-v0.4 report not found.")

V03_RUN = V03_RUNS[-1]
V04_RUN = V04_RUNS[-1]

with open(V03_RUN/"report.json","r",encoding="utf-8") as f:
    v03_report = json.load(f)
with open(V04_RUN/"report.json","r",encoding="utf-8") as f:
    v04_report = json.load(f)

selected_configs = v03_report["feedback_configs"]
if not selected_configs:
    raise ValueError("No frozen feedback configurations found.")

print("Source v0.3:", V03_RUN)
print("Source v0.4:", V04_RUN)
print("Frozen feedback configs:")
for c in selected_configs:
    print(c)


Source v0.3: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/error-amplification-v03-20260815-201450
Source v0.4: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/statistical-mechanism-audit-v04-20260815-202739
Frozen feedback configs:
{'method': 'mean', 'k': 20, 'alpha': 0.3, 'temperature': None}
{'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature': 0.1}


## 2. Seal protocol before loading DEV qrels or running DEV retrieval


In [5]:
protocol = {
    "schema_version": 1,
    "status": "SEALED_BEFORE_DEV_RETRIEVAL",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "FEVER",
    "confirmation_split": "DEV",
    "expected_dev_query_count": 6666,
    "encoder": "BAAI/bge-small-en-v1.5",
    "corpus_rows": 5_416_568,
    "dimension": DIM,
    "top_retrieve": TOP_RETRIEVE,
    "top_k": TOP_K,
    "feedback_rounds": MAX_ROUNDS,
    "nlist": NLIST,
    "nprobe": NPROBE,
    "conditions": {
        "low_fidelity": "IVF-PQ32",
        "high_fidelity": "IVF-SQ8",
    },
    "feedback_configs": selected_configs,
    "primary_endpoints": [
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
        "H4_B_minus_A",
    ],
    "primary_unit": "query",
    "method_pooling": "average within query across all previously frozen feedback configs",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "randomization_samples": RANDOMIZATION_SAMPLES,
    "multiple_testing": "Holm jointly across H1-H4",
    "endpoint_pass_rule": "mean>0 and bootstrap_CI_low>0 and Holm_p<0.05",
    "full_confirmation_rule": "all H1-H4 pass",
    "source_v03": str(V03_RUN),
    "source_v04": str(V04_RUN),
    "dev_based_selection_allowed": False,
    "test_retrieval_allowed": False,
    "test_qrels_access_allowed": False,
}

canonical = json.dumps(
    protocol,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",",":"),
).encode("utf-8")

protocol_sha256 = hashlib.sha256(canonical).hexdigest()
protocol["protocol_sha256"] = protocol_sha256

PROTOCOL_PATH = OUT/"sealed_protocol.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

(OUT/"SEALED_BEFORE_DEV_RETRIEVAL.txt").write_text(
    "SEALED\n"
    f"protocol_sha256={protocol_sha256}\n"
    f"created_at_utc={protocol['created_at_utc']}\n",
    encoding="utf-8",
)

print("SEALED:", PROTOCOL_PATH)
print("SHA-256:", protocol_sha256)


SEALED: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/sealed_protocol.json
SHA-256: 6873cc8f341893fde8fe3ec5f6998d3137340c5d2f4a44bf716cbbe3c19548fc


## 3. Load corpus/query mapping and DEV relevance


In [6]:
size_bytes = CORPUS_MEMMAP.stat().st_size
bytes_per_row = DIM * np.dtype(np.float16).itemsize
assert size_bytes % bytes_per_row == 0

n_docs = size_bytes // bytes_per_row
assert n_docs == 5_416_568

docs = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(n_docs,DIM),
)

query_embeddings = np.load(QUERY_EMB, mmap_mode="r")
assert query_embeddings.shape == (123_142,DIM)

with open(QUERY_IDS,"r",encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

assert len(query_ids) == len(query_embeddings)
query_row = {qid:i for i,qid in enumerate(query_ids)}

with open(SPLIT_MANIFEST,"r",encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x).strip() for x in split["dev_query_ids"]]
test_ids = [str(x).strip() for x in split["test_query_ids"]]

assert len(dev_ids) == 6666
assert all(q in query_row for q in dev_ids)
assert split["test_retrieval_performed"] is False
assert split["test_relevance_values_accessed"] is False

dev_rows = np.array([query_row[q] for q in dev_ids], dtype=np.int64)

dev_qrels_df = pd.read_csv(DEV_QRELS)
dev_qrels_df["query-id"] = dev_qrels_df["query-id"].astype(str)

assert set(dev_qrels_df["query-id"]).issubset(set(dev_ids))
assert dev_qrels_df["corpus-row"].between(0,n_docs-1).all()

dev_qrels = {}
for qid,g in dev_qrels_df.groupby("query-id"):
    rel = g[g["score"]>0]["corpus-row"].astype(np.int64)
    dev_qrels[str(qid)] = set(rel.tolist())

print("DEV queries:", len(dev_ids))
print("DEV qrels queries:", len(dev_qrels))
print("TEST membership known only:", len(test_ids))


DEV queries: 6666
DEV qrels queries: 6666
TEST membership known only: 6666


## 4. Locate frozen cached indexes


In [7]:
def unique_index(pattern):
    matches = sorted(CACHE_ROOT.glob(pattern), key=lambda p:p.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(pattern)
    return matches[-1]

PQ32_PATH = unique_index(
    "fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
)
SQ8_PATH = unique_index(
    "fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss"
)

print("PQ32:", PQ32_PATH)
print("SQ8 :", SQ8_PATH)


PQ32: /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss
SQ8 : /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss


## 5. Metric and feedback helpers


In [8]:
def normalize_rows(x):
    x=np.asarray(x,np.float32)
    n=np.linalg.norm(x,axis=1,keepdims=True)
    return x/np.maximum(n,1e-12)

def evaluate_batch(qids, ranked_ids, k=10):
    recall=np.empty(len(qids),np.float32)
    mrr=np.empty(len(qids),np.float32)
    ndcg=np.empty(len(qids),np.float32)

    discounts=1.0/np.log2(np.arange(2,k+2))

    for i,qid in enumerate(qids):
        relset=dev_qrels.get(str(qid),set())
        ranked=ranked_ids[i,:k]
        hits=np.array(
            [1.0 if int(d) in relset else 0.0 for d in ranked],
            dtype=np.float32,
        )

        recall[i]=hits.sum()/max(len(relset),1)

        pos=np.flatnonzero(hits)
        mrr[i]=1.0/(pos[0]+1) if len(pos) else 0.0

        dcg=float((hits*discounts).sum())
        ideal=min(len(relset),k)
        idcg=float(discounts[:ideal].sum()) if ideal else 0.0
        ndcg[i]=dcg/idcg if idcg>0 else 0.0

    return recall,mrr,ndcg

def jaccard_rows(a,b):
    vals=np.empty(len(a),np.float32)
    for i in range(len(a)):
        A=set(map(int,a[i]))
        B=set(map(int,b[i]))
        vals[i]=len(A&B)/max(len(A|B),1)
    return vals

def cosine_distance_rows(a,b):
    a=normalize_rows(a)
    b=normalize_rows(b)
    return 1.0-np.sum(a*b,axis=1)

def feedback_matrix(ids,scores,config,batch_size=256):
    k=int(config["k"])
    out=np.empty((len(ids),DIM),np.float32)

    for start in range(0,len(ids),batch_size):
        end=min(start+batch_size,len(ids))
        dids=np.asarray(ids[start:end,:k],dtype=np.int64)
        x=np.asarray(docs[dids],dtype=np.float32)

        if config["method"]=="mean":
            f=x.mean(axis=1)
        elif config["method"]=="softmax":
            temp=float(config["temperature"])
            z=np.asarray(scores[start:end,:k],np.float64)/temp
            z-=z.max(axis=1,keepdims=True)
            w=np.exp(np.clip(z,-60,60))
            w/=np.maximum(w.sum(axis=1,keepdims=True),1e-12)
            f=(x*w[:,:,None]).sum(axis=1)
        else:
            raise ValueError(config["method"])

        f=f.astype(np.float32)
        f/=np.maximum(np.linalg.norm(f,axis=1,keepdims=True),1e-12)
        out[start:end]=f

    return out

def anchored_update_matrix(q0,feedback,alpha):
    q=((1.0-float(alpha))*q0+float(alpha)*feedback).astype(np.float32)
    q/=np.maximum(np.linalg.norm(q,axis=1,keepdims=True),1e-12)
    return q

def config_key(config):
    temp="none" if config.get("temperature") is None else str(config["temperature"]).replace(".","p")
    return (
        f"{config['method']}"
        f"-k{int(config['k'])}"
        f"-a{str(config['alpha']).replace('.','p')}"
        f"-t{temp}"
    )


## 6. Synchronized PQ32 / SQ8 DEV trajectories


In [9]:
DEV_Q0 = normalize_rows(
    np.asarray(query_embeddings[dev_rows], dtype=np.float32)
)

def run_self_feedback(index_path,condition,config):
    index=faiss.read_index(str(index_path))
    index.nprobe=NPROBE

    q0=DEV_Q0
    qt=q0.copy()
    states=[]

    for t in range(MAX_ROUNDS+1):
        print(condition,config_key(config),"iteration",t)

        scores,ids=index.search(
            np.ascontiguousarray(qt,np.float32),
            TOP_RETRIEVE,
        )

        recall,mrr,ndcg=evaluate_batch(dev_ids,ids,TOP_K)

        states.append({
            "q":qt.copy(),
            "ids":ids.copy(),
            "scores":scores.copy(),
            "recall":recall,
            "mrr":mrr,
            "ndcg":ndcg,
        })

        if t<MAX_ROUNDS:
            fb=feedback_matrix(ids,scores,config)
            qt=anchored_update_matrix(q0,fb,config["alpha"])

    del index
    gc.collect()
    return states

state_cache={}

for config in selected_configs:
    ck=config_key(config)

    for condition,path in [
        ("ivfpq32",PQ32_PATH),
        ("ivfsq8",SQ8_PATH),
    ]:
        cache_path=OUT/f"states-{condition}-{ck}.npz"

        if cache_path.is_file():
            z=np.load(cache_path)
            states=[]
            for t in range(MAX_ROUNDS+1):
                states.append({
                    "q":z[f"q_{t}"],
                    "ids":z[f"ids_{t}"],
                    "scores":z[f"scores_{t}"],
                    "recall":z[f"recall_{t}"],
                    "mrr":z[f"mrr_{t}"],
                    "ndcg":z[f"ndcg_{t}"],
                })
            print("Loaded:",cache_path)
        else:
            states=run_self_feedback(path,condition,config)
            payload={}
            for t,s in enumerate(states):
                for field in ["q","ids","scores","recall","mrr","ndcg"]:
                    payload[f"{field}_{t}"]=s[field]
            np.savez_compressed(cache_path,**payload)
            print("Saved:",cache_path)

        state_cache[(condition,ck)]=states
        ram_status(f"{condition} {ck}")

print("Synchronized trajectories complete.")


ivfpq32 mean-k20-a0p3-tnone iteration 0
ivfpq32 mean-k20-a0p3-tnone iteration 1
ivfpq32 mean-k20-a0p3-tnone iteration 2
ivfpq32 mean-k20-a0p3-tnone iteration 3
ivfpq32 mean-k20-a0p3-tnone iteration 4
Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/states-ivfpq32-mean-k20-a0p3-tnone.npz
[RAM] ivfpq32 mean-k20-a0p3-tnone  RSS=  1.01 GB | available= 10.95 GB | used= 13.5%
ivfsq8 mean-k20-a0p3-tnone iteration 0
ivfsq8 mean-k20-a0p3-tnone iteration 1
ivfsq8 mean-k20-a0p3-tnone iteration 2
ivfsq8 mean-k20-a0p3-tnone iteration 3
ivfsq8 mean-k20-a0p3-tnone iteration 4
Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/states-ivfsq8-mean-k20-a0p3-tnone.npz
[RAM] ivfsq8 mean-k20-a0p3-tnone   RSS=  1.38 GB | available= 10.58 GB | used= 16.5%
ivfpq32 softmax-k5-a0p5-t0p1 iteration 0
ivfpq32 softmax-k5-a0p5-t0p1 iteration 1
ivfpq32 softmax-k5-a0p5-t0p1 iteration 2
ivfpq32 softmax-k5-a0p5

## 7. Construct H1–H3 query-level endpoints


In [10]:
trajectory_rows=[]

for config in selected_configs:
    ck=config_key(config)
    A=state_cache[("ivfpq32",ck)]
    B=state_cache[("ivfsq8",ck)]

    candidate_t0=None

    for t in range(MAX_ROUNDS+1):
        dq=cosine_distance_rows(A[t]["q"],B[t]["q"])
        dc=1.0-jaccard_rows(A[t]["ids"],B[t]["ids"])

        if t==0:
            candidate_t0=dc.copy()

        dc_inc=dc-candidate_t0
        ugap=B[t]["ndcg"]-A[t]["ndcg"]
        augap=np.abs(ugap)

        for i,qid in enumerate(dev_ids):
            trajectory_rows.append({
                "query_id":qid,
                "method":config["method"],
                "config_key":ck,
                "iteration":t,
                "query_divergence":float(dq[i]),
                "candidate_divergence_increment":float(dc_inc[i]),
                "abs_utility_gap":float(augap[i]),
                "pq32_ndcg":float(A[t]["ndcg"][i]),
                "sq8_ndcg":float(B[t]["ndcg"][i]),
            })

trajectory_df=pd.DataFrame(trajectory_rows)
trajectory_df.to_parquet(
    OUT/"sealed_dev_paired_trajectories.parquet",
    index=False,
)

def linear_slope(g,metric):
    g=g.sort_values("iteration")
    x=g["iteration"].to_numpy(np.float64)
    y=g[metric].to_numpy(np.float64)
    return float(np.polyfit(x,y,1)[0])

method_slope_rows=[]

for (qid,method,ck),g in trajectory_df.groupby(
    ["query_id","method","config_key"]
):
    method_slope_rows.append({
        "query_id":qid,
        "method":method,
        "config_key":ck,
        "H1_query_divergence_slope":linear_slope(g,"query_divergence"),
        "H2_candidate_increment_slope":linear_slope(g,"candidate_divergence_increment"),
        "H3_abs_utility_gap_slope":linear_slope(g,"abs_utility_gap"),
    })

method_slopes=pd.DataFrame(method_slope_rows)

query_slopes=(
    method_slopes
    .groupby("query_id",as_index=False)
    [[
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
    ]]
    .mean()
)

display(query_slopes.describe())


,H1_query_divergence_slope,H2_candidate_increment_slope,H3_abs_utility_gap_slope
count,6666.000000,6666.000000,6666.000000
mean,0.004109,0.008157,0.007261
std,0.002924,0.015130,0.033909
min,-0.001095,-0.128925,-0.250889
25%,0.002050,-0.000108,0.000000
50%,0.003453,0.007929,0.000000
75%,0.005427,0.016552,0.000000
max,0.023758,0.065370,0.300000


## 8. H4 frozen feedback-source intervention


In [11]:
def run_intervention(config):
    pq32=faiss.read_index(str(PQ32_PATH))
    sq8=faiss.read_index(str(SQ8_PATH))
    pq32.nprobe=NPROBE
    sq8.nprobe=NPROBE

    q0=DEV_Q0
    qA=q0.copy()
    qB=q0.copy()

    rows=[]

    for t in range(MAX_ROUNDS+1):
        print("intervention",config_key(config),"iteration",t)

        sA,idA=pq32.search(np.ascontiguousarray(qA,np.float32),TOP_RETRIEVE)
        sB_search,idB_search=pq32.search(np.ascontiguousarray(qB,np.float32),TOP_RETRIEVE)
        sB_fb,idB_fb=sq8.search(np.ascontiguousarray(qB,np.float32),TOP_RETRIEVE)

        rA,mA,nA=evaluate_batch(dev_ids,idA,TOP_K)
        rB,mB,nB=evaluate_batch(dev_ids,idB_search,TOP_K)

        for i,qid in enumerate(dev_ids):
            rows.append({
                "query_id":qid,
                "method":config["method"],
                "config_key":config_key(config),
                "iteration":t,
                "A_ndcg":float(nA[i]),
                "B_ndcg":float(nB[i]),
            })

        if t<MAX_ROUNDS:
            fA=feedback_matrix(idA,sA,config)
            fB=feedback_matrix(idB_fb,sB_fb,config)
            qA=anchored_update_matrix(q0,fA,config["alpha"])
            qB=anchored_update_matrix(q0,fB,config["alpha"])

    del pq32,sq8
    gc.collect()
    return pd.DataFrame(rows)

intervention_frames=[]

for config in selected_configs:
    ck=config_key(config)
    cache_path=OUT/f"intervention-{ck}.parquet"

    if cache_path.is_file():
        df=pd.read_parquet(cache_path)
        print("Loaded:",cache_path)
    else:
        df=run_intervention(config)
        df.to_parquet(cache_path,index=False)
        print("Saved:",cache_path)

    intervention_frames.append(df)

intervention_df=pd.concat(intervention_frames,ignore_index=True)

final=intervention_df[
    intervention_df["iteration"]==MAX_ROUNDS
].copy()

final["H4_B_minus_A"]=final["B_ndcg"]-final["A_ndcg"]

method_intervention=final[
    ["query_id","method","config_key","H4_B_minus_A"]
].copy()

query_intervention=(
    method_intervention
    .groupby("query_id",as_index=False)["H4_B_minus_A"]
    .mean()
)

display(query_intervention.describe())


intervention mean-k20-a0p3-tnone iteration 0
intervention mean-k20-a0p3-tnone iteration 1
intervention mean-k20-a0p3-tnone iteration 2
intervention mean-k20-a0p3-tnone iteration 3
intervention mean-k20-a0p3-tnone iteration 4
Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/intervention-mean-k20-a0p3-tnone.parquet
intervention softmax-k5-a0p5-t0p1 iteration 0
intervention softmax-k5-a0p5-t0p1 iteration 1
intervention softmax-k5-a0p5-t0p1 iteration 2
intervention softmax-k5-a0p5-t0p1 iteration 3
intervention softmax-k5-a0p5-t0p1 iteration 4
Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/intervention-softmax-k5-a0p5-t0p1.parquet


,H4_B_minus_A
count,6666.000000
mean,0.030376
std,0.113373
min,-0.670932
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


## 9. Frozen confirmatory inference


In [12]:
def bootstrap_mean_chunked(values,samples=BOOTSTRAP_SAMPLES,seed=0,chunk=RESAMPLE_CHUNK):
    x=np.asarray(values,np.float64)
    x=x[np.isfinite(x)]
    n=len(x)

    rng=np.random.default_rng(seed)
    draws=[]
    remaining=samples

    while remaining>0:
        b=min(chunk,remaining)
        idx=rng.integers(0,n,size=(b,n),dtype=np.int32)
        draws.append(x[idx].mean(axis=1))
        remaining-=b

    draws=np.concatenate(draws)

    return {
        "n":int(n),
        "mean":float(x.mean()),
        "median":float(np.median(x)),
        "ci_low":float(np.quantile(draws,.025)),
        "ci_high":float(np.quantile(draws,.975)),
    }

def sign_flip_chunked(values,samples=RANDOMIZATION_SAMPLES,seed=0,chunk=RESAMPLE_CHUNK):
    x=np.asarray(values,np.float64)
    x=x[np.isfinite(x)]

    n=len(x)
    obs=float(x.mean())
    rng=np.random.default_rng(seed)

    exceed=0
    done=0

    while done<samples:
        b=min(chunk,samples-done)
        signs=rng.integers(0,2,size=(b,n),dtype=np.int8)
        signs=signs.astype(np.float32)*2.0-1.0
        stats=(signs*x[None,:]).mean(axis=1)
        exceed+=int(np.sum(stats>=obs))
        done+=b

    return (exceed+1)/(samples+1)

def holm_adjust(pvalues):
    p=np.asarray(pvalues,np.float64)
    m=len(p)
    order=np.argsort(p)
    adjusted=np.empty(m,np.float64)
    running=0.0

    for rank,idx in enumerate(order):
        raw=(m-rank)*p[idx]
        running=max(running,raw)
        adjusted[idx]=min(running,1.0)

    return adjusted

endpoint_values={
    "H1_query_divergence_slope":
        query_slopes["H1_query_divergence_slope"].to_numpy(np.float64),
    "H2_candidate_increment_slope":
        query_slopes["H2_candidate_increment_slope"].to_numpy(np.float64),
    "H3_abs_utility_gap_slope":
        query_slopes["H3_abs_utility_gap_slope"].to_numpy(np.float64),
    "H4_B_minus_A":
        query_intervention["H4_B_minus_A"].to_numpy(np.float64),
}

rows=[]

for i,(name,x) in enumerate(endpoint_values.items()):
    boot=bootstrap_mean_chunked(x,seed=SEED+1000+i)
    p=sign_flip_chunked(x,seed=SEED+2000+i)

    rows.append({
        "endpoint":name,
        **boot,
        "randomization_p":p,
        "fraction_positive":float(np.mean(x>0)),
        "fraction_zero":float(np.mean(np.isclose(x,0,atol=1e-12))),
        "fraction_negative":float(np.mean(x<0)),
    })

confirm_df=pd.DataFrame(rows)
confirm_df["holm_p"]=holm_adjust(confirm_df["randomization_p"])

confirm_df["pass"]=(
    (confirm_df["mean"]>0)
    & (confirm_df["ci_low"]>0)
    & (confirm_df["holm_p"]<0.05)
)

display(confirm_df)


,endpoint,n,mean,median,ci_low,ci_high,randomization_p,fraction_positive,fraction_zero,fraction_negative,holm_p,pass
0,H1_query_divergence_slope,6666,0.004109,0.003453,0.004039,0.004179,0.00005,0.996100,0.000000,0.003900,0.0002,True
1,H2_candidate_increment_slope,6666,0.008157,0.007929,0.007796,0.008515,0.00005,0.744374,0.000000,0.255626,0.0002,True
2,H3_abs_utility_gap_slope,6666,0.007261,0.000000,0.006454,0.008061,0.00005,0.103960,0.875338,0.030303,0.0002,True
3,H4_B_minus_A,6666,0.030376,0.000000,0.027680,0.033097,0.00005,0.103210,0.890789,0.006001,0.0002,True


## 10. Secondary method-specific robustness


In [13]:
secondary=[]

for method,g in method_slopes.groupby("method"):
    for metric in [
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
    ]:
        x=(
            g.groupby("query_id")[metric]
            .mean()
            .to_numpy(np.float64)
        )
        b=bootstrap_mean_chunked(x,samples=10_000,seed=SEED+3000)
        secondary.append({
            "method":method,
            "endpoint":metric,
            **b,
            "fraction_positive":float(np.mean(x>0)),
        })

for method,g in method_intervention.groupby("method"):
    x=(
        g.groupby("query_id")["H4_B_minus_A"]
        .mean()
        .to_numpy(np.float64)
    )
    b=bootstrap_mean_chunked(x,samples=10_000,seed=SEED+4000)
    secondary.append({
        "method":method,
        "endpoint":"H4_B_minus_A",
        **b,
        "fraction_positive":float(np.mean(x>0)),
    })

secondary_df=pd.DataFrame(secondary)
display(secondary_df)


,method,endpoint,n,mean,median,ci_low,ci_high,fraction_positive
0,mean,H1_query_divergence_slope,6666,0.001056,0.000749,0.001031,0.001082,0.997150
1,mean,H2_candidate_increment_slope,6666,0.005054,0.004496,0.004669,0.005423,0.663366
2,mean,H3_abs_utility_gap_slope,6666,0.005349,0.000000,0.004533,0.006195,0.073357
3,softmax,H1_query_divergence_slope,6666,0.007162,0.005992,0.007034,0.007292,0.989349
4,softmax,H2_candidate_increment_slope,6666,0.011259,0.010816,0.010785,0.011727,0.748125
5,softmax,H3_abs_utility_gap_slope,6666,0.009174,0.000000,0.008198,0.010177,0.097510
6,mean,H4_B_minus_A,6666,0.013065,0.000000,0.010904,0.015264,0.044254
7,softmax,H4_B_minus_A,6666,0.047687,0.000000,0.043491,0.051932,0.096610


## 11. Descriptive FIT→DEV effect preservation


In [14]:
fit_reference={
    "H1_query_divergence_slope":0.003694,
    "H2_candidate_increment_slope":0.007529,
    "H3_abs_utility_gap_slope":0.005672,
    "H4_B_minus_A":0.024726,
}

preservation=[]

for _,r in confirm_df.iterrows():
    ref=fit_reference[r["endpoint"]]
    preservation.append({
        "endpoint":r["endpoint"],
        "FIT_reference":ref,
        "DEV_effect":float(r["mean"]),
        "DEV_over_FIT":float(r["mean"]/ref),
        "same_direction":bool(np.sign(r["mean"])==np.sign(ref)),
    })

preservation_df=pd.DataFrame(preservation)
display(preservation_df)


,endpoint,FIT_reference,DEV_effect,DEV_over_FIT,same_direction
0,H1_query_divergence_slope,0.003694,0.004109,1.112349,True
1,H2_candidate_increment_slope,0.007529,0.008157,1.083357,True
2,H3_abs_utility_gap_slope,0.005672,0.007261,1.280212,True
3,H4_B_minus_A,0.024726,0.030376,1.228501,True


## 12. Sealed verdict


In [15]:
all_pass=bool(confirm_df["pass"].all())

print("=== ARC-v0.5 SEALED FEVER DEV CONFIRMATION ===")
print("Protocol SHA-256:",protocol_sha256)
print()
display(
    confirm_df[
        [
            "endpoint","mean","ci_low","ci_high",
            "randomization_p","holm_p",
            "fraction_positive","pass"
        ]
    ]
)
print()
display(preservation_df)

if all_pass:
    decision=(
        "SEALED CONFIRMATION PASS: all four preregistered approximation-feedback "
        "amplification endpoints replicate on untouched FEVER DEV under query-level "
        "inference and Holm family-wise correction. Proceed to cross-dataset and "
        "cross-approximation replication."
    )
else:
    failed=confirm_df.loc[~confirm_df["pass"],"endpoint"].tolist()
    decision=(
        "SEALED CONFIRMATION FAIL: failed endpoints: "
        + ", ".join(failed)
        + ". Do not broaden the mechanism claim before diagnosis."
    )

print("DECISION:",decision)


=== ARC-v0.5 SEALED FEVER DEV CONFIRMATION ===
Protocol SHA-256: 6873cc8f341893fde8fe3ec5f6998d3137340c5d2f4a44bf716cbbe3c19548fc



,endpoint,mean,ci_low,ci_high,randomization_p,holm_p,fraction_positive,pass
0,H1_query_divergence_slope,0.004109,0.004039,0.004179,0.00005,0.0002,0.996100,True
1,H2_candidate_increment_slope,0.008157,0.007796,0.008515,0.00005,0.0002,0.744374,True
2,H3_abs_utility_gap_slope,0.007261,0.006454,0.008061,0.00005,0.0002,0.103960,True
3,H4_B_minus_A,0.030376,0.027680,0.033097,0.00005,0.0002,0.103210,True


,endpoint,FIT_reference,DEV_effect,DEV_over_FIT,same_direction
0,H1_query_divergence_slope,0.003694,0.004109,1.112349,True
1,H2_candidate_increment_slope,0.007529,0.008157,1.083357,True
2,H3_abs_utility_gap_slope,0.005672,0.007261,1.280212,True
3,H4_B_minus_A,0.024726,0.030376,1.228501,True


DECISION: SEALED CONFIRMATION PASS: all four preregistered approximation-feedback amplification endpoints replicate on untouched FEVER DEV under query-level inference and Holm family-wise correction. Proceed to cross-dataset and cross-approximation replication.


## 13. Save immutable confirmation evidence


In [16]:
query_slopes.to_csv(OUT/"dev_query_pooled_slopes.csv",index=False)
method_slopes.to_csv(OUT/"dev_query_method_slopes.csv",index=False)
query_intervention.to_csv(OUT/"dev_query_pooled_intervention.csv",index=False)
method_intervention.to_csv(OUT/"dev_query_method_intervention.csv",index=False)
confirm_df.to_csv(OUT/"confirmatory_endpoints.csv",index=False)
secondary_df.to_csv(OUT/"secondary_method_robustness.csv",index=False)
preservation_df.to_csv(OUT/"fit_dev_effect_preservation.csv",index=False)

report={
    "status":"SEALED_FEVER_DEV_CONFIRMATION_COMPLETE",
    "protocol_sha256":protocol_sha256,
    "source_protocol":str(PROTOCOL_PATH),
    "dev_query_count":len(dev_ids),
    "feedback_configs":selected_configs,
    "primary_endpoints":confirm_df.to_dict(orient="records"),
    "effect_preservation":preservation_df.to_dict(orient="records"),
    "all_primary_endpoints_pass":all_pass,
    "decision":decision,
    "dev_based_selection_performed":False,
    "test_retrieval_performed":False,
    "test_qrels_accessed":False,
    "completed_at_utc":datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH=OUT/"report.json"
REPORT_PATH.write_text(
    json.dumps(report,indent=2,ensure_ascii=False,default=float),
    encoding="utf-8",
)

report_sha=hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
(OUT/"FINAL_REPORT_SHA256.txt").write_text(report_sha+"\n",encoding="utf-8")

print("Saved:",OUT)
print("Report:",REPORT_PATH)
print("Report SHA-256:",report_sha)


Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214
Report: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/sealed-fever-dev-confirmation-v05/20260816-072214/report.json
Report SHA-256: baa28d64f3b8f1cc5a9bb8b36db1f6058108706ed2e284d9539018b4b26665c5
